In [ ]:
# Imports
import cProfile
import pstats
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization.wcsaxes import WCSAxes
from astropy.coordinates import SkyCoord, FK5
from spectral_cube import SpectralCube
from velocity_tools import extract_streamline, gradient_descent, stream_lines_grad
from velocity_tools import stream_lines # won't use this directly, but needed to compare with stream_lines_grad
import os
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import value_and_grad
import pandas as pd
import optax

import warnings
warnings.filterwarnings('ignore', message='.*PV2_1.*')
warnings.filterwarnings('ignore', message='.*PV2_2.*')
warnings.filterwarnings('ignore', message='.*TIMESYS.*')

# Settings
hltau_c= SkyCoord("4h31m38.43s", "+18d13m57.19s", frame='fk5')
hltau_ref = hltau_c.skyoffset_frame()
iras2a_c = SkyCoord("3h28m55.569s", "+31d14m37.025s", frame='fk5')
iras2a_ref = iras2a_c.skyoffset_frame()
distance_hltau = 147 #parsecs
distance_iras2a = 293 #parsecs
# choose which distance
distance = distance_iras2a

# cubefile = 'test_data/HLTau/HLTAU_HCOp32.fits'a
# file_Tpeak = 'test_data/HLTau/HLTAU_HCOp32_Tpeak.fits'
cubefile = 'test_data/IRAS2A/D2CO_streamer_cluster_data.fits'
file_Tpeak = 'test_data/IRAS2A/D2CO_streamer_cluster_tpeak.fits'

# some constants
G = 6.67430e-11 * (1e-3)**2 * (1.988416e30) / (1.4959787e11) # in au (km/s)^2 * Msol^-1
au_in_km = 1.4959787e8 #km



### Original cube with spectra: Prepare the 1D streamer emission from the cube

In [ ]:
# get the spectralcube object from the data using spectral-cube
hdu = fits.open(cubefile)[0]
cube = SpectralCube.read(hdu).with_spectral_unit(u.km/u.s, rest_value=hdu.header['RESTFRQ']*u.Hz)

# TODO: this streamer extraction should be replaced with clustering-based streamer extraction

# extract the subcube with the streamer (we got this from the tipsy tutorial, no need to plot)
## Limits for extracting subcube with streamer

'''
vmin = 7    # Min. vel. of streamer
vmax = 10
xmin = -3   # Min R.A. offset (arcsec) to consider for streamer
xmax = -1
ymin = -3   # Min. Decl. offset (arcsec) to consider for streamer
ymax = 0.5
rms_thresh = 4  # sigma threshold for streamer
'''

vmin = 6
vmax = 8
xmin = -5
xmax = 5
ymin = -12
ymax = 0.5
rms_thresh = 4

# extract the streamer subcube
info_header = cube.header  # header of the cube, contains required information
x_conv_fac = 1/60/60/info_header['CDELT1']
xmin_p = int((xmin*x_conv_fac)+info_header['CRPIX1'])
xmax_p = int((xmax*x_conv_fac)+info_header['CRPIX1'])
y_conv_fac = 1/60/60/info_header['CDELT2']
ymin_p = int((ymin*y_conv_fac)+info_header['CRPIX2'])
ymax_p = int((ymax*y_conv_fac)+info_header['CRPIX2'])
vunit = cube.spectral_axis.unit
## Note: a check can be added to see if requested limits are within the limits of the cube itself

streamer_cubev = cube.spectral_slab(vmin*vunit,vmax*vunit)    # Selecting velocities
streamer_cubevc = streamer_cubev[:,min(ymin_p,ymax_p):max(ymin_p,ymax_p)   
                    ,min(xmin_p,xmax_p):max(xmin_p,xmax_p)]   # Selecting pixels
#     print(min(ymin_p,ymax_p),max(ymin_p,ymax_p),min(xmin_p,xmax_p),max(xmin_p,xmax_p)) 
streamer_cube = streamer_cubevc.with_mask(streamer_cubevc > rms_thresh*streamer_cubevc.mad_std())  # Removing low flux values 

print('vunit:', vunit)
print('Streamer cube spectral axis limits:', streamer_cube.spectral_axis.min(), streamer_cube.spectral_axis.max())


In [ ]:
n_points = 10 # the number of points we want to reduce the data to


# Extract 1D streamline from the data cube
pc_coords, pc_means, pc_stds = extract_streamline.reduce_to_1D(streamer_cube, n_elements=n_points)
print(f"point cloud velocities (km/s): {pc_coords[2]}")

# Prepare data for gradient descent
ra_data = pc_means[0] # offsets in arcsec
dec_data = pc_means[1] # offsets in arcsec
v_data = pc_means[2]   # velocities in km/s (rel to vlsr)
print(f"data velocities (km/s): {v_data}")

ra_sigma = pc_stds[0]
dec_sigma = pc_stds[1]
v_sigma = pc_stds[2]

rproj = np.sqrt(ra_data**2 + dec_data**2)
print(f"projected distances from star (arcsec): {rproj}")

data = (ra_data, dec_data, v_data)
uncertainties = (ra_sigma, dec_sigma, v_sigma)

In [ ]:
# Helper function to compute and plot radial bin edges
def get_radial_bin_partitions(pc_coords, n_elements):
    """
    Compute the radial bin partition boundaries used in reduce_to_1D.
    
    Parameters
    ----------
    pc_coords : array of shape (3, n_points)
        Point cloud coordinates. Index 0 = RA, Index 1 = Dec, Index 2 = velocity
    n_elements : int
        Number of elements used in the reduction
    
    Returns
    -------
    partitions : array of shape (n_elements + 1,)
        Radial bin boundaries in arcsec
    """
    import numpy as np
    ra_coords = pc_coords[0]
    dec_coords = pc_coords[1]
    # Compute radial distance metric (same as in extract_streamline.get_distance_metric)
    distance_metric = np.sqrt(ra_coords**2 + dec_coords**2)
    # Compute percentile boundaries
    b_per = np.linspace(0, 100, n_elements + 1)
    partitions = np.array([np.percentile(distance_metric, per) for per in b_per])
    return partitions

def plot_radial_bin_circles(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5):
    """
    Plot circles showing the radial bin edges on an RA-Dec plot.
    Axis limits are preserved after adding the circles.
    
    Parameters
    ----------
    ax : matplotlib axes object
        The axes to plot on
    partitions : array
        Radial bin boundaries in arcsec
    color : str
        Color of the circles
    linewidth : float
        Line width of the circles
    alpha : float
        Transparency of the circles
    """
    import matplotlib.patches as patches
    # Save current axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    
    # Add circles
    for partition in partitions:
        circle = patches.Circle((0, 0), partition, fill=False, edgecolor=color, 
                               linewidth=linewidth, alpha=alpha)
        ax.add_patch(circle)
    
    # Restore original axis limits
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

In [ ]:
# plot it
import matplotlib.pyplot as plt
# plot the observed data points as a scatter
plt.scatter(pc_coords[0], pc_coords[1], s=1, alpha=0.3, color='grey', label='Point cloud')
# plot the extracted 1D streamline with error bars
plt.errorbar(ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma, fmt='o-', label='Extracted 1D Streamline', color='red')
# plot the star
plt.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
# plot the radial bin edges
partitions = get_radial_bin_partitions(pc_coords, n_points)
ax = plt.gca()
plot_radial_bin_circles(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5)
plt.xlabel('RA Offset (arcsec)')
plt.ylabel('Dec Offset (arcsec)')
# flip the x axis to match the astronomical convention (RA increases to the left)
plt.gca().invert_xaxis()
plt.legend()
plt.title('Extracted 1D Streamer emission')


### Initial guess at parameters, and plot

The plot is so you can refine your initial parameters a bit. The comparison between stream_lines and stream_lines_grad is just a sanity check and will be removed

In [ ]:

# Initial guesses at params

# IRAS 2A D2CO

r0 = 1500*u.au
theta0 = 40.*u.deg
phi0 = 100.*u.deg
omega0 = 5e-13/u.s
v_r0 = 1.0*u.km/u.s
#params we set
Mstar = 4.0*u.Msun
inc = -45*u.deg
PA_ang = 194*u.deg #228
v_lsr = 7.5*u.km/u.s

# Run the forward model with initial params - using stream_lines
# x = RA offset (au), y = velocity (km/s), z = Dec offset (au)
# print("Running stream_lines...")
(x1, y1, z1), (vx1, vy1, vz1) = stream_lines.xyz_stream(
                mass=Mstar, r0=r0, theta0=theta0, phi0=phi0,
                omega=omega0, v_r0=v_r0, inc=inc, pa=PA_ang,
                rmin=0.5e2*u.au, deltar=40*u.au) #<- decrease deltar for more accurate streamer calculation
dra_stream1 = -x1.value / distance * u.arcsec 
ddec_stream1 = z1.value / distance * u.arcsec
fil1 = SkyCoord(dra_stream1, ddec_stream1, frame=iras2a_ref).transform_to(FK5)

# print("---------------------------------------------")

# Run the forward model with initial params - using stream_lines_grad
# convert angles to radians, and strip units from all quantities
r0 = r0.to(u.au).value
theta0 = theta0.to(u.rad).value
phi0 = phi0.to(u.rad).value
omega0 = omega0.to(1/u.s).value
v_r0 = v_r0.to(u.km/u.s).value
inc = inc.to(u.rad).value
PA_ang = PA_ang.to(u.rad).value
Mstar = Mstar.to(u.Msun).value
v_lsr = v_lsr.to(u.km/u.s).value
# x = RA offset (au), y = velocity (km/s), z = Dec offset (au)
# print("Running stream_lines_grad...")
(x2, y2, z2), (vx2, vy2, vz2) = stream_lines_grad.xyz_stream(
                mass=Mstar, r0=r0, theta0=theta0, phi0=phi0,
                omega=omega0, v_r0=v_r0, inc=inc, pa=PA_ang,
                rmin=0.5e2, deltar=40) #<- decrease deltar for more accurate streamer calculation
dra_stream2 = -np.array(x2, dtype=np.float64) / distance * u.arcsec # note the _value to get raw number for stream_lines_grad
ddec_stream2 = np.array(z2, dtype=np.float64) / distance * u.arcsec
fil2 = SkyCoord(dra_stream2, ddec_stream2, frame=iras2a_ref).transform_to(FK5)

# print("Plotting results...")
# Load the moment 0 data
hdu = fits.open(file_Tpeak)[0]
tpeak_data = hdu.data.squeeze()
wcs_Tpeak = WCS(hdu.header)

# Create figure with two subplots side by side
plt.close('all')
fig = plt.figure(figsize=(14, 6))

# # Left subplot - stream_lines
ax1 = WCSAxes(fig, [0.05, 0.1, 0.4, 0.8], wcs=wcs_Tpeak.celestial)
fig.add_axes(ax1)
im1 = ax1.imshow(tpeak_data, cmap='inferno', vmin=0)
ax1.scatter(iras2a_c.ra, iras2a_c.dec, marker='*', transform=ax1.get_transform('world'),
    facecolor='white', edgecolor='black')
ax1.plot(fil1.ra, fil1.dec, color='black', transform=ax1.get_transform('world'), linewidth=5)
ax1.plot(fil1.ra, fil1.dec, color='red', transform=ax1.get_transform('world'), linewidth=2)
ax1.set_title('stream_lines')

# Right subplot - stream_lines_grad
ax2 = WCSAxes(fig, [0.52, 0.1, 0.4, 0.8], wcs=wcs_Tpeak.celestial)
fig.add_axes(ax2)
im2 = ax2.imshow(tpeak_data, cmap='inferno', vmin=0)
ax2.scatter(iras2a_c.ra, iras2a_c.dec, marker='*', transform=ax2.get_transform('world'),
     facecolor='white', edgecolor='black')
ax2.plot(fil2.ra, fil2.dec, color='black', transform=ax2.get_transform('world'), linewidth=5)
ax2.plot(fil2.ra, fil2.dec, color='red', transform=ax2.get_transform('world'), linewidth=2)
ax2.set_title('stream_lines_grad')


plt.show()



### TEST Forward model and calculating loss (with extra plots)

First set your input params in the format required

In [ ]:
# Parameters to optimize
initial_opt_params = {
    'r0': 1500.0,  # au
    'theta0': 40.0,  # degrees
    'phi0': 100.0,  # degrees
    'log_omega': np.log(5e-13),  # log(1/s)
    'v_r0': 1.0,  # km/s
}

# Fixed parameters (not optimized)
# HL Tau
# pa = 138 + 270 = 408 degrees, which is equivalent to 48 degrees (since PA is modulo 360)

'''
fixed_params = {
    'mass': 2.1,  # solar masses
    'inc': -47.0,  # degrees
    'pa': 48.0,  # degrees
    'rmin': 50.0,  # au
    'deltar': 40.0,  # au
    'v_lsr': 7.1  # km/s (systemic velocity)
}
'''

# IRAS2A
fixed_params = {
    'mass': 4.0,  # solar masses
    'inc': -45.0,  # degrees
    'pa': 194.0,  # degrees
    'rmin': 50.0,  # au
    'deltar':50.0,  # au
    'v_lsr': 7.5  # km/s (systemic velocity)
}


# Convert angles from degrees to radians
initial_opt_params['theta0'] = np.radians(initial_opt_params['theta0'])
initial_opt_params['phi0'] = np.radians(initial_opt_params['phi0'])
fixed_params['inc'] = np.radians(fixed_params['inc'])
fixed_params['pa'] = np.radians(fixed_params['pa'])

Doing chi2 loss more manually, just so we can check the model point choosing is working properly (delete this later)

In [ ]:
# forward model (same inputs as chi2_loss)
ra_model, dec_model, v_model = gradient_descent.forward_model(
    initial_opt_params, fixed_params, distance
)

# keep finite points for plotting only
finite_model = jnp.isfinite(ra_model) & jnp.isfinite(dec_model) & jnp.isfinite(v_model)
ra_model_plot = ra_model[finite_model]
dec_model_plot = dec_model[finite_model]
v_model_plot = v_model[finite_model]

# match model to data exactly as chi2_loss does
ra_model_interp, dec_model_interp, v_model_interp, valid, _dmetric_model, _overlap_min, _overlap_max = gradient_descent.match_model_to_data_curve(
    ra_model, dec_model, v_model, ra_data, dec_data
)

print(initial_opt_params)
print(fixed_params)
print(f"ra_data = {ra_data}")
print(f"ra_model_interp = {ra_model_interp}")
print(f"retained {int(jnp.sum(valid))}/{len(valid)} data points after overlap filtering")

# ---- Manual chi2_loss calculation (same logic as gradient_descent.chi2_loss) ----
# Coerce to float64 and floor sigmas to avoid division by zero
ra_data_f = jnp.asarray(ra_data, dtype=jnp.float64)
dec_data_f = jnp.asarray(dec_data, dtype=jnp.float64)
v_data_f = jnp.asarray(v_data, dtype=jnp.float64)

ra_sigma_f = jnp.asarray(ra_sigma, dtype=jnp.float64)
dec_sigma_f = jnp.asarray(dec_sigma, dtype=jnp.float64)
v_sigma_f = jnp.asarray(v_sigma, dtype=jnp.float64)

eps = jnp.asarray(1e-8, dtype=jnp.float64)
ra_sigma_safe = jnp.maximum(ra_sigma_f, eps)
dec_sigma_safe = jnp.maximum(dec_sigma_f, eps)
v_sigma_safe = jnp.maximum(v_sigma_f, eps)

# Smooth overlap barrier used by the optimizer

def softplus_barrier(value, tau):
    tau = jnp.asarray(tau, dtype=jnp.float64)
    return tau * jnp.logaddexp(jnp.asarray(0.0, dtype=jnp.float64), jnp.asarray(value, dtype=jnp.float64) / tau)

# Recompute distance metrics and overlap domain exactly like chi2_loss
dmetric_data = extract_streamline.get_distance_metric(ra_data_f, dec_data_f)
dmetric_model = extract_streamline.get_distance_metric(ra_model, dec_model)

model_finite = jnp.isfinite(dmetric_model)
data_finite = jnp.isfinite(dmetric_data)

model_min = jnp.min(jnp.where(model_finite, dmetric_model, jnp.inf))
model_max = jnp.max(jnp.where(model_finite, dmetric_model, -jnp.inf))
data_min = jnp.min(jnp.where(data_finite, dmetric_data, jnp.inf))
data_max = jnp.max(jnp.where(data_finite, dmetric_data, -jnp.inf))

overlap_min = jnp.maximum(model_min, data_min)
overlap_max = jnp.minimum(model_max, data_max)

margin = jnp.asarray(0.5, dtype=jnp.float64)
tau = jnp.asarray(0.05, dtype=jnp.float64)

penalty = (
    softplus_barrier(overlap_min - dmetric_data, tau)
    + softplus_barrier(dmetric_data - overlap_max, tau)
)
chi2_penalty = jnp.sum((penalty / margin) ** 2)

# Polar-space sky residual and velocity residual
r_data, theta_data = extract_streamline.cartesian_to_polar(ra_data_f, dec_data_f)
_, theta_model = extract_streamline.cartesian_to_polar(ra_model_interp, dec_model_interp)

dtheta = extract_streamline._wrap_to_pi(theta_data - theta_model)
dsky = r_data * dtheta
sigma_dsky = jnp.sqrt(ra_sigma_safe**2 + dec_sigma_safe**2)

chi2_dsky = jnp.sum((dsky / sigma_dsky) ** 2)
chi2_v = jnp.sum(((v_data_f - v_model_interp) / v_sigma_safe) ** 2)
chi2_total = chi2_dsky + chi2_v + chi2_penalty

print(
    f"Chi2 dsky: {chi2_dsky:.2f}, Chi2 v: {chi2_v:.2f}, "
    f"Chi2 penalty: {chi2_penalty:.2f}, Total: {chi2_total:.2f}"
)
print(f"Overlap range in distance metric: [{float(overlap_min):.4f}, {float(overlap_max):.4f}]")
# ---- Plot ----
import matplotlib.pyplot as plt
plt.scatter(pc_coords[0], pc_coords[1], s=1, color='grey', alpha=0.3, label='Point cloud')
plt.errorbar(ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma, fmt='o-', label='Extracted 1D Streamline', color='red')
plt.plot(ra_model_plot, dec_model_plot, 'b-', linewidth=2, label='Model Streamline')

# matched model/data points in retained overlap
plt.scatter(
    ra_model_interp[valid], dec_model_interp[valid],
    s=25, label='Model at retained data arc lengths', color='blue', zorder=5
)
plt.scatter(
    ra_data[valid], dec_data[valid],
    s=45, facecolor='none', edgecolor='cyan', linewidth=1.2,
    label='Retained data points', zorder=6
)

plt.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
partitions = get_radial_bin_partitions(pc_coords, n_points)
ax = plt.gca()
plot_radial_bin_circles(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5)
plt.gca().invert_xaxis()
plt.xlabel('RA Offset (arcsec)')
plt.ylabel('Dec Offset (arcsec)')
plt.legend()

## Full fit streamline starts here

In [ ]:
def get_omega(mass, r0):
    '''
    this gets value of omega when r_cent = 0.5 * r0
    '''
    omega_squared = 0.5 * G * mass / (jnp.power(r0, 3) * jnp.power(au_in_km, 2)) # in s^-2
    omega = jnp.power(omega_squared, 0.5) # in s^-1
    return omega

opt_params = initial_opt_params.copy()

# Define physically reasonable bounds (omega bounds transformed to natural log space)
# These bounds are also used as normalization anchors: x_norm = (x - min) / (max - min).
# Provide bounds for every optimized parameter.
r0_min, r0_max = 200.0, 20000.0 # param bounds in au

# the omega bounds are set by keeping centrifugal radius reasonable (r_cent = 0.5 r0)
omega_max = get_omega(fixed_params['mass'], r0_min)
omega_min = get_omega(fixed_params['mass'], r0_max)
# print these in scientific notation for sanity check
print(f"Omega bounds: {omega_min:.2e} to {omega_max:.2e} 1/s")

param_bounds = {
    'r0': (r0_min, r0_max),                    # radius between 200-20000 au
    'theta0': (0.0, np.pi),                    # polar angle 0-pi
    'phi0': (0.0, 2*np.pi),                    # azimuthal angle 0-2pi
    'log_omega': (np.log(omega_min), np.log(omega_max)),  # omega in [omega_min, omega_max] 1/s
    'v_r0': (-10.0, 10.0),                       # radial velocity 0 to 2 km/s
}

log_file = 'streamfit_test_output/optimisation_log.csv'
trace_file = 'streamfit_test_output/optimisation_trace.csv'
trace_every = 1
n_epochs = 500
info_every = 10
learning_rate = 0.005 # Single learning rate applied to all normalized optimization parameters
loss_method = 'rthetavel' # options: 'radecvel', 'rthetavel'

gradient_tol = 1e-2 * len(initial_opt_params) # gradient tolerance scaled by number of parameters

## here we run the fit, using cProfile to track performance
profile = False # set to True to enable cProfile profiling of the optimization run
if profile:
    profiler = cProfile.Profile()
    profiler.enable()

best_opt_params, loss_history, param_errors = gradient_descent.fit_streamline(
    opt_params,
    fixed_params,
    data,
    uncertainties,
    distance,
    learning_rate=learning_rate,
    param_bounds=param_bounds,
    n_epochs=n_epochs,
    info_every=info_every,
    loss_threshold=0.05,
    loss_threshold_epochs=20,
    gradient_tol=gradient_tol,
    gradient_tol_epochs=20,
    early_stopping_patience=200,
    log_file=log_file,
    trace_file=trace_file,
    trace_every=trace_every,
    loss_method=loss_method,
    output_uncertainties=True,
 )

if profile:
    profiler.disable() # Stop profiling after optimization is complete


print(f"Optimized using loss_method='{loss_method}'")

# Plot loss history
# Epoch indexing: epoch 0 = initial state, epoch i (i >= 1) = after update i
# loss_history is 0-indexed: loss_history[i] = loss at epoch i
plt.figure(figsize=(8, 5))
epochs = range(len(loss_history))
plt.plot(epochs, loss_history, marker='o', markersize=4)
plt.xlabel('Epoch (0 = initial, i = after update i)')
plt.ylabel('Loss')
plt.title('Optimization Progress\n(Epoch i = loss after applying i updates; epoch 0 = initial)')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

## Best fit visualisation

In [ ]:
# optionally we can also plot the by eye streamline
add_by_eye = True
# by eye parameters
by_eye_params = {
    'r0': 2540.0,  # au
    'theta0': 54.0,  # degrees
    'phi0': 61.0,  # degrees
    'log_omega': np.log(7e-13),  # log(1/s)
    'v_r0': 0.0,  # km/s
}
# convert angles to radians for forward model
by_eye_params['theta0'] = np.radians(by_eye_params['theta0'])
by_eye_params['phi0'] = np.radians(by_eye_params['phi0'])
if add_by_eye:
    ra_by_eye, dec_by_eye, v_by_eye = gradient_descent.forward_model(by_eye_params, fixed_params, distance)



# Final model with best-fit parameters
ra_best, dec_best, v_best = gradient_descent.forward_model(best_opt_params, fixed_params, distance)

# remove NaN values (due to rmin) from model for plotting
not_nan = ~jnp.isnan(ra_best) & ~jnp.isnan(dec_best) & ~jnp.isnan(v_best)
ra_best = ra_best[not_nan]
dec_best = dec_best[not_nan]
v_best = v_best[not_nan]

fig = plt.figure(figsize=(6.5, 7))
# plot it on top of the data
plt.scatter(pc_coords[0], pc_coords[1], s=1, color='gray', alpha=0.3, label='Point cloud', zorder=4)
plt.errorbar(ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma, fmt='o-', label='Extracted 1D Streamline', color='red', zorder=5)
plt.plot(ra_best, dec_best, color='blue', linewidth=2, label='Best-fit', zorder=7)

# optionally plot by-eye model
if add_by_eye:
    plt.plot(ra_by_eye, dec_by_eye, color='tab:green', linewidth=2, label='By-eye', zorder=6)

# get the model positions at data arc length points (overlap restricted)
ra_best_interp, dec_best_interp, v_best_interp, valid, _dmetric_model, _overlap_min, _overlap_max = gradient_descent.match_model_to_data_curve(
    ra_best, dec_best, v_best, ra_data, dec_data)

# plot model positions only where overlap is retained
plt.scatter(
    ra_best_interp[valid], dec_best_interp[valid],
    s=25, color='blue', zorder=6
)
# plt.scatter(
#     ra_data[valid], dec_data[valid],
#     s=45, facecolor='none', edgecolor='cyan', linewidth=1.2,
#     label='Retained data points', zorder=6
# )

plt.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', zorder=10)
plt.xlabel('RA Offset (arcsec)')
plt.ylabel('Dec Offset (arcsec)')

# Save axis limits before adding background/circles
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Fill background grey, then cut out the partition circles with white fill
import matplotlib.patches as patches
ax.set_facecolor('lightgrey')

partitions = get_radial_bin_partitions(pc_coords, n_points)

# Draw filled white circles for each partition to reveal the data area inside
for partition_radius in partitions:
    circle = patches.Circle((0, 0), partition_radius, 
                           facecolor='white', 
                           edgecolor='none', 
                           zorder=1)
    ax.add_patch(circle)

plot_radial_bin_circles(ax, partitions, color='gray', linewidth=1, alpha=0.3)

# Restore original axis limits
ax.set_xlim(xlim)
ax.set_ylim(ylim)

plt.gca().invert_xaxis()
plt.legend()
# plt.title('Best-fit Streamline Model')
plt.show()

## d(Loss)/d(parameter)

In [ ]:
# Compute gradient of the loss with respect to each optimizable parameter for each epoch
# Load the optimisation log produced by the fit
opt_log = pd.read_csv("streamfit_test_output/optimisation_log.csv")
epochs = opt_log['epoch'].values

# Get the list of optimizable parameter names (excluding epoch and loss)
param_names = [col for col in opt_log.columns if col not in ("epoch", "loss")]

# Initialize dictionary to store gradients for each parameter
gradients = {param: [] for param in param_names}

# helper: make a loss function of a single scalar parameter for a fixed epoch row
def _make_param_loss(row, param_name):
    def _f(param_value):
        # Create params dict from row
        params = {k: float(row[k]) for k in param_names}
        # Override with the parameter being differentiated
        params[param_name] = param_value
        return gradient_descent.chi2_loss(params, fixed_params, data, uncertainties, distance, prepared_data=extract_streamline.prepare_data(data, uncertainties), loss_method='rthetavel')
    return _f

# Compute gradients for all parameters at all epochs
for _, row in opt_log.iterrows():
    for param_name in param_names:
        f_param = _make_param_loss(row, param_name)
        g = jax.grad(f_param)(jnp.array(row[param_name], dtype=jnp.float64))
        gradients[param_name].append(float(g))

# Display gradients
print("Gradients computed for all optimizable parameters:")
for param_name in param_names:
    print(f"\n{param_name}:")
    print(gradients[param_name])

In [ ]:
# Plot the gradient vs epoch for all parameters
fig, axes = plt.subplots(len(param_names), 1, figsize=(10, 3 * len(param_names)))
if len(param_names) == 1:
    axes = [axes]  # Make it a list for consistency

for idx, param_name in enumerate(param_names):
    ax = axes[idx]
    ax.plot(epochs, gradients[param_name], linewidth=2)
    ax.set_ylabel(f'd(Loss)/d({param_name})')
    ax.grid(True, alpha=0.4)
    ax.axhline(0, color='k', linestyle='--', alpha=0.6)

axes[-1].set_xlabel('Epoch')
fig.suptitle('Gradients of loss with respect to all parameters per epoch')
fig.tight_layout()
plt.show()

## Uncertainty visualisations

In [ ]:
# Uncertainty visualizations: diagonal error bars, parameter correlation, and streamline spaghetti
import numpy as np
import matplotlib.pyplot as plt

# Use only optimized parameters (exclude derived omega from best_opt_params if present)
opt_keys = list(initial_opt_params.keys())
best_for_cov = {k: float(best_opt_params[k]) for k in opt_keys}

# Prepare data-only quantities once
prepared_data = extract_streamline.prepare_data(data, uncertainties)

# Recover covariance from Hessian-based uncertainty estimate
param_errors_cov, cov = gradient_descent.estimate_parameter_errors(
    best_for_cov,
    fixed_params,
    data,
    uncertainties,
    distance,
    prepared_data,
    loss_method=loss_method,
    gradient_tol=gradient_tol,
    normalization_spec=None,
    )

param_errors_plot = {k: float(param_errors[k]) for k in opt_keys}

# ---------- 1) Normalized parameter error bars ----------
param_vals = np.array([best_for_cov[k] for k in opt_keys], dtype=float)
param_errs = np.array([param_errors_plot[k] for k in opt_keys], dtype=float)

# Avoid divide-by-zero issues
eps = 1e-12

# Relative (fractional) errors
norm_errs = param_errs / (np.abs(param_vals) + eps)

fig, ax = plt.subplots(figsize=(8, 4.5))

ypos = np.arange(len(opt_keys))

ax.barh(
    ypos,
    norm_errs,
    color='tab:blue',
    alpha=0.8
)

ax.set_yticks(ypos)
ax.set_yticklabels(opt_keys)

ax.set_xlabel('Relative uncertainty ($\\sigma / |x|$)')
ax.set_title('Normalized Parameter Uncertainties')

ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

# ---------- 2) Correlation heatmap from covariance ----------
# normalised correlation_{i,j} = covariance_{i,j} / (sigma_i * sigma_j)
cov_np = np.array(cov, dtype=float)
print("Covariance matrix:")
print(cov_np)
diag = np.sqrt(np.clip(np.diag(cov_np), 1e-30, None))
corr = cov_np / np.outer(diag, diag)
corr = np.clip(corr, -1.0, 1.0)

from mpl_toolkits.axes_grid1 import make_axes_locatable

fig, ax = plt.subplots(figsize=(6.5, 5.5))

im = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm_r')

ax.set_xticks(np.arange(len(opt_keys)))
ax.set_yticks(np.arange(len(opt_keys)))
ax.set_xticklabels(opt_keys, rotation=45, ha='right', fontsize=11)
ax.set_yticklabels(opt_keys, fontsize=11)
ax.set_title('Parameter Correlation Matrix')

# Create colorbar axis with matched height
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="5%", pad=0.08)

cbar = fig.colorbar(im, cax=cax)
cbar.set_label('Correlation coefficient')

for i in range(len(opt_keys)):
    for j in range(len(opt_keys)):
        ax.text(
            j, i,
            f'{corr[i, j]:.2f}',
            ha='center',
            va='center',
            fontsize=10,
            color='black'
        )

plt.tight_layout()
plt.show()

In [ ]:
# ---------- 3) Streamline spaghetti from covariance sampling ----------
rng = np.random.default_rng(7)
mu = np.array([best_for_cov[k] for k in opt_keys], dtype=float)

n_samples = 100
samples = rng.multivariate_normal(mu, cov_np, size=n_samples)

# Clipping to user bounds if present
for j, key in enumerate(opt_keys):
    if key in param_bounds:
        lo, hi = param_bounds[key]
        samples[:, j] = np.clip(samples[:, j], lo, hi)

fig, (ax_sky, ax_v) = plt.subplots(1, 2, figsize=(10, 5))

# Plot sampled streamlines
for s in samples:
    sample_params = {k: float(v) for k, v in zip(opt_keys, s)}
    try:
        ra_s, dec_s, v_s = gradient_descent.forward_model(sample_params, fixed_params, distance)
        ra_s = np.array(ra_s, dtype=float)
        dec_s = np.array(dec_s, dtype=float)
        v_s = np.array(v_s, dtype=float)
        finite = np.isfinite(ra_s) & np.isfinite(dec_s) & np.isfinite(v_s)
        if np.sum(finite) < 3:
            continue

        ra_f = ra_s[finite]
        dec_f = dec_s[finite]
        v_f = v_s[finite]
        d_f = np.array(extract_streamline.get_distance_metric(ra_f, dec_f), dtype=float)
        ord_idx = np.argsort(d_f)

        ax_sky.plot(ra_f, dec_f, color='tab:blue', alpha=0.06, lw=1)
        ax_v.plot(d_f[ord_idx], v_f[ord_idx], color='tab:blue', alpha=0.06, lw=1)
    except Exception:
        # Skip pathological sampled parameter combinations
        continue

# Overlay best-fit streamline
ra_best_plot = np.array(ra_best, dtype=float)
dec_best_plot = np.array(dec_best, dtype=float)
v_best_plot = np.array(v_best, dtype=float)
d_best = np.array(extract_streamline.get_distance_metric(ra_best_plot, dec_best_plot), dtype=float)
ord_best = np.argsort(d_best)

ax_sky.plot(ra_best_plot, dec_best_plot, color='blue', lw=2, label='Best-fit')
ax_sky.errorbar(
    ra_data, dec_data, xerr=ra_sigma, yerr=dec_sigma,
    fmt='o', color='red', ecolor='red', ms=4, alpha=0.9, label='Data'
    )
ax_sky.invert_xaxis()
ax_sky.set_xlabel('RA Offset (arcsec)')
ax_sky.set_ylabel('Dec Offset (arcsec)')
ax_sky.set_title('Sky-plane Streamline Spaghetti')
ax_sky.legend()

ax_v.plot(d_best[ord_best], v_best_plot[ord_best], color='blue', lw=2, label='Best-fit')
d_data = np.array(extract_streamline.get_distance_metric(ra_data, dec_data), dtype=float)
ord_data = np.argsort(d_data)
ax_v.errorbar(
    d_data[ord_data], np.array(v_data)[ord_data], yerr=np.array(v_sigma)[ord_data],
    fmt='o', color='red', ecolor='red', ms=4, alpha=0.9, label='Data'
    )
ax_v.set_xlabel('Projected distance (arcsec)')
ax_v.set_ylabel('Velocity (km/s)')
ax_v.set_title('Velocity Spaghetti')
ax_v.legend()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from astropy import units as u
import pandas as pd
import os
from astropy.io import fits
from spectral_cube import SpectralCube
from velocity_tools import extract_streamline, gradient_descent, stream_lines_grad
import jax
import jax.numpy as jnp

# Helper functions for plotting radial bin edges
def get_radial_bin_partitions(pc_coords, n_elements):
    """
    Compute the radial bin partition boundaries used in reduce_to_1D.
    
    Parameters
    ----------
    pc_coords : array of shape (3, n_points)
        Point cloud coordinates. Index 0 = RA, Index 1 = Dec, Index 2 = velocity
    n_elements : int
        Number of elements used in the reduction
    
    Returns
    -------
    partitions : array of shape (n_elements + 1,)
        Radial bin boundaries in arcsec
    """
    import numpy as np
    ra_coords = pc_coords[0]
    dec_coords = pc_coords[1]
    # Compute radial distance metric (same as in extract_streamline.get_distance_metric)
    distance_metric = np.sqrt(ra_coords**2 + dec_coords**2)
    # Compute percentile boundaries
    b_per = np.linspace(0, 100, n_elements + 1)
    partitions = np.array([np.percentile(distance_metric, per) for per in b_per])
    return partitions

def plot_radial_bin_circles(ax, partitions, color='lightgrey', linewidth=1, alpha=0.5):
    """
    Plot circles showing the radial bin edges on an RA-Dec plot.
    Axis limits are preserved after adding the circles.
    
    Parameters
    ----------
    ax : matplotlib axes object
        The axes to plot on
    partitions : array
        Radial bin boundaries in arcsec
    color : str
        Color of the circles
    linewidth : float
        Line width of the circles
    alpha : float
        Transparency of the circles
    """
    import matplotlib.patches as patches
    # Save current axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    
    # Add circles
    for partition in partitions:
        circle = patches.Circle((0, 0), partition, fill=False, edgecolor=color, 
                               linewidth=linewidth, alpha=alpha)
        ax.add_patch(circle)
    
    # Restore original axis limits
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

In [ ]:
# function to find spikes in the loss
def find_spikes(loss, threshold=0.1):
    spikes = []
    for i in range(1, len(loss) - 1):
        if loss[i] > loss[i - 1] * (1 + threshold) and loss[i] > loss[i + 1] * (1 + threshold):
            spikes.append(i)
    return spikes

In [ ]:
# import data
optimisation_log = pd.read_csv("streamfit_test_output/optimisation_log.csv")
trace_log_path = "streamfit_test_output/optimisation_trace.csv"
if os.path.exists(trace_log_path):
    trace_log = pd.read_csv(trace_log_path)
    print(f"Loaded tracer log: {trace_log_path} ({len(trace_log)} rows)")
else:
    trace_log = None
    print(f"Tracer log not found at {trace_log_path}. Re-run test_streamfit.ipynb with trace_file enabled.")

if trace_log is not None:
    if {"chi2_ra", "chi2_dec", "chi2_v"}.issubset(trace_log.columns):
        trace_loss_method = "radecvel"
        trace_component_cols = ["chi2_ra", "chi2_dec", "chi2_v"]
    elif {"chi2_r", "chi2_theta", "chi2_v"}.issubset(trace_log.columns):
        trace_loss_method = "rthetavel"
        trace_component_cols = ["chi2_r", "chi2_theta", "chi2_v"]
    else:
        trace_loss_method = "unknown"
        trace_component_cols = []
    print(f"Detected trace loss method: {trace_loss_method}")
else:
    trace_loss_method = None
    trace_component_cols = []

optimisation_log

epochs = optimisation_log["epoch"].values
loss = optimisation_log["loss"].values

lowest_loss = min(loss)
best_epoch = epochs[loss.argmin()]

spikes = find_spikes(loss, threshold=0.1)
spike_epochs = epochs[spikes]
print(f"Spikes found at epochs: {spike_epochs.tolist()}")

plt.plot(epochs, loss)
plt.yscale("log")
plt.scatter(best_epoch, lowest_loss, color="green", label=f"Best Epoch: {best_epoch}\n Loss: {lowest_loss:.0f}")
plt.scatter(spike_epochs, loss[spikes], color="orange", label="Spikes")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")

We want to investigate which change in parameters causes the spikes in loss

In [ ]:
# plot the parameters over epochs in a single figure with subplots
# Epoch semantics: epoch 0 = initial, epoch i (i >= 1) = after update i
param_names = [col for col in optimisation_log.columns if col not in ("epoch", "loss")]

fig, axes = plt.subplots(len(param_names) + 2, 1, figsize=(8, 3 * (len(param_names) + 1)), sharex=True)
loss_ax = axes[0]
loss_ax.plot(epochs, loss, color="black")
loss_ax.scatter(best_epoch, lowest_loss, color="green", label=f"Best Epoch: {best_epoch}\n Loss: {lowest_loss:.0f}")
loss_ax.scatter(epochs[spikes], loss[spikes], color="orange", label="Spikes")
loss_ax.set_ylabel("loss (epoch 0=initial, i>=1=after update i)")
loss_ax.set_yscale("log")
loss_ax.grid(True)
loss_ax.legend()

# plot chi^2 components based on available trace columns
chi2_ax = axes[1]
if trace_log is None:
    raise FileNotFoundError("No optimisation trace log found. Run test_streamfit.ipynb with trace_file enabled first.")
if not trace_component_cols:
    raise ValueError(
        "Trace log does not contain a recognized chi2 component set. "
        "Expected either [chi2_ra, chi2_dec, chi2_v] or [chi2_r, chi2_theta, chi2_v]."
    )

for col in trace_component_cols:
    chi2_ax.plot(epochs, trace_log[col].values, label=col.replace("_", " "))

chi2_best_col = trace_component_cols[0]
chi2_ax.scatter(best_epoch, trace_log[chi2_best_col].values[loss.argmin()], color="green", label="Best Epoch")
chi2_ax.scatter(epochs[spikes], trace_log[chi2_best_col].values[spikes], color="orange", label="Spikes")
chi2_ax.set_ylabel("chi2")
chi2_ax.set_yscale("log")
chi2_ax.set_title(f"Loss components ({trace_loss_method})")
chi2_ax.grid(True)
chi2_ax.legend()

for ax, param in zip(axes[2:], param_names):
    param_values = optimisation_log[param].values
    ax.plot(epochs, param_values)
    ax.scatter(best_epoch, param_values[loss.argmin()], color="green", label="Best Epoch")
    ax.scatter(epochs[spikes], param_values[spikes], color="orange", label="Spikes")
    ax.set_ylabel(param)
    ax.grid(True)
    ax.legend()
    ax.set_xlabel("Epoch (0=initial, i>=1=after update i)")
fig.tight_layout()
plt.show()

In [ ]:
# plot tracer diagnostics over epochs and mark loss-spike epochs
# Epoch semantics: epoch 0 = initial, epoch i (i >= 1) = after update i
if trace_log is None:
    raise FileNotFoundError("No optimisation trace log found. Run test_streamfit.ipynb with trace_file enabled first.")

trace_log = trace_log.sort_values("epoch").reset_index(drop=True)
trace_cols = [col for col in trace_log.columns if col not in ("epoch", "loss")]

if not trace_cols:
    raise ValueError("No tracer columns found in optimisation trace log.")

spike_epoch_set = set(int(e) for e in spike_epochs.tolist())
trace_spike_mask = trace_log["epoch"].astype(int).isin(spike_epoch_set)

fig, axes = plt.subplots(len(trace_cols) + 1, 1, figsize=(10, 2.5 * (len(trace_cols) + 1)), sharex=True)

loss_ax = axes[0]
loss_ax.plot(epochs, loss, color="black")
loss_ax.scatter(best_epoch, lowest_loss, color="green", label=f"Best Epoch: {best_epoch}\n Loss: {lowest_loss:.0f}")
loss_ax.scatter(spike_epochs, loss[spikes], color="orange", label="Spikes")
loss_ax.set_ylabel("loss (epoch 0=initial, i>=1=after update i)")
loss_ax.set_yscale("log")
loss_ax.grid(True, alpha=0.3)
loss_ax.legend()

for ax, col in zip(axes[1:], trace_cols):
    ax.plot(trace_log["epoch"].values, trace_log[col].values, color="tab:blue")
    if trace_spike_mask.any():
        ax.scatter(
            trace_log.loc[trace_spike_mask, "epoch"].values,
            trace_log.loc[trace_spike_mask, col].values,
            color="orange",
            s=18,
            label="Spikes",
            zorder=5,
        )
        ax.legend()
    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Epoch (0=initial, i>=1=after update i)")
fig.tight_layout()
plt.show()

Plot the model at every epoch

In [ ]:
# --------------------------------------------------
# Precompute models per epoch
# (fixed_params and distance are reused from earlier initialization)
# --------------------------------------------------
epoch_models = []

for idx, epoch in enumerate(epochs):
    row = optimisation_log.iloc[idx]
    opt_params_epoch = {param: float(row[param]) for param in param_names}

    ra_model, dec_model, v_model = gradient_descent.forward_model(
        opt_params_epoch, fixed_params, distance
    )

    not_nan = ~jnp.isnan(ra_model) & ~jnp.isnan(dec_model) & ~jnp.isnan(v_model)
    ra_model = ra_model[not_nan]
    dec_model = dec_model[not_nan]
    v_model = v_model[not_nan]

    ra_model_interp, dec_model_interp, v_model_interp, valid, _, _, _ = \
        gradient_descent.match_model_to_data_curve(
            ra_model, dec_model, v_model, ra_data, dec_data
        )

    epoch_models.append({
        'epoch': epoch,
        'opt_params_epoch': opt_params_epoch,
        'ra_model': ra_model,
        'dec_model': dec_model,
        'v_model': v_model,
        'ra_model_interp': ra_model_interp,
        'dec_model_interp': dec_model_interp,
        'v_model_interp': v_model_interp,
        'valid': valid,
        'v_data_valid': v_data[valid],
        'ra_sigma_valid': ra_sigma[valid],
        'dec_sigma_valid': dec_sigma[valid],
        'v_sigma_valid': v_sigma[valid],
    })

# --------------------------------------------------
# Compute global axis limits (for consistent frames)
# --------------------------------------------------
all_ra = []
all_dec = []

for epoch_data in epoch_models:
    all_ra.extend(epoch_data['ra_model'])
    all_dec.extend(epoch_data['dec_model'])

# include observational data
all_ra.extend(ra_data)
all_dec.extend(dec_data)

# include point cloud
all_ra.extend(pc_coords[0])
all_dec.extend(pc_coords[1])

all_ra = jnp.array(all_ra)
all_dec = jnp.array(all_dec)

mask = ~jnp.isnan(all_ra) & ~jnp.isnan(all_dec)
all_ra = all_ra[mask]
all_dec = all_dec[mask]

ra_min, ra_max = float(all_ra.min()), float(all_ra.max())
dec_min, dec_max = float(all_dec.min()), float(all_dec.max())

# padding (5%)
pad_ra = 0.05 * (ra_max - ra_min)
pad_dec = 0.05 * (dec_max - dec_min)

# NOTE: RA axis inverted later
ra_lim = (ra_min - pad_ra, ra_max + pad_ra)
dec_lim = (dec_min - pad_dec, dec_max + pad_dec)

# --------------------------------------------------
# Output directory
# --------------------------------------------------
output_dir = "streamfit_test_output/epochs"
os.makedirs(output_dir, exist_ok=True)
# clear existing contents
for filename in os.listdir(output_dir):
    file_path = os.path.join(output_dir, filename)
    if os.path.isfile(file_path):
        os.remove(file_path)

# --------------------------------------------------
# Plot and save each epoch separately
# --------------------------------------------------
for idx, epoch_data in enumerate(epoch_models):
    fig, ax = plt.subplots(figsize=(6, 6))

    ra_model = epoch_data['ra_model']
    dec_model = epoch_data['dec_model']
    ra_model_interp = epoch_data['ra_model_interp']
    dec_model_interp = epoch_data['dec_model_interp']
    valid = epoch_data['valid']
    epoch = epoch_data['epoch']

    if idx == 0:
        print(epoch_data['opt_params_epoch'])
        print(fixed_params)
        print(ra_model_interp[:5], dec_model_interp[:5])

    # Plot elements
    ax.scatter(pc_coords[0], pc_coords[1], s=1, alpha=0.3,
               color='grey', label='Point cloud')

    ax.errorbar(ra_data, dec_data,
                xerr=ra_sigma,
                yerr=dec_sigma,
                fmt='o-', color='red',
                label='Extracted 1D Streamline')

    ax.plot(ra_model, dec_model,
            color='blue', linewidth=2,
            label='Model Streamline')

    ax.scatter(ra_model_interp[valid], dec_model_interp[valid],
               s=25, color='blue', zorder=5)
    ax.scatter(
        ra_data[valid], dec_data[valid],
        s=45, facecolor='none', edgecolor='cyan', linewidth=1.2, zorder=6
    )

    ax.scatter(0, 0, marker='*', s=100,
               color='yellow', edgecolor='black', zorder=10)

    plot_radial_bin_circles(
        ax,
        get_radial_bin_partitions(pc_coords, n_points),
        color='lightgrey',
        linewidth=1,
        alpha=0.5
    )

    # Consistent axes
    ax.set_xlim(ra_lim)
    ax.set_ylim(dec_lim)
    ax.invert_xaxis()

    ax.set_xlabel('RA Offset (arcsec)')
    ax.set_ylabel('Dec Offset (arcsec)')
    ax.set_title(f"Epoch: {int(epoch)}")

    ax.legend(loc='upper left')

    # Save frame
    save_path = os.path.join(output_dir, f"epoch_{int(epoch):03d}.png")
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close(fig)

In [ ]:
# turn them into a video using ffmpeg
import subprocess

fps = 5

input_pattern = os.path.join(output_dir, "epoch_%03d.png")
output_video = os.path.join(output_dir, "streamline_evolution.mp4")

ffmpeg_cmd = [
    "ffmpeg",
    "-y",
    "-framerate", str(fps),
    "-i", input_pattern,
    "-vf",
    "setpts='PTS/(1+0.01*N)',pad=ceil(iw/2)*2:ceil(ih/2)*2",
    "-pix_fmt", "yuv420p",
    output_video
]

try:
    subprocess.run(ffmpeg_cmd, check=True)
    print(f"Video saved to {output_video}")
except subprocess.CalledProcessError as e:
    print(f"Error creating video: {e}")
except FileNotFoundError:
    print("ffmpeg not found. Please install ffmpeg to create the video.")

In [ ]:
# plot ra vs velocity for every epoch in a single figure with subplots
if 'epoch_models' not in globals():
    raise RuntimeError("Run Cell 8 first to precompute epoch_models.")

# Calculate grid dimensions
num_epochs = len(epochs)
n_cols = 4
n_rows = (num_epochs + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()
for idx, epoch_data in enumerate(epoch_models):
    ax = axes[idx]

    ra_model = epoch_data['ra_model']
    v_model = epoch_data['v_model']
    ra_model_interp = epoch_data['ra_model_interp']
    v_model_interp = epoch_data['v_model_interp']
    valid = epoch_data['valid']
    epoch = epoch_data['epoch']

    ax.scatter(pc_coords[0], pc_coords[2], s=1, alpha=0.3, color='grey', label='Point cloud')
    ax.scatter(0, 7, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
    ax.errorbar(
        ra_data[valid], v_data[valid],
        xerr=ra_sigma[valid], yerr=v_sigma[valid],
        fmt='o-', label='Retained data', color='red'
    )
    ax.plot(ra_model, v_model, color='blue', linewidth=2, label='Model Streamline')
    ax.scatter(
        ra_model_interp[valid], v_model_interp[valid],
        s=25, label='Model at retained data arc lengths', color='blue', zorder=5
    )
    ax.scatter(
        ra_data[valid], v_data[valid],
        s=45, facecolor='none', edgecolor='cyan', linewidth=1.2,
        label='Retained data points', zorder=6
    )
    ax.set_xlabel('RA Offset (arcsec)')
    ax.set_ylabel('Velocity (km/s)')
    ax.set_title('RA vs Velocity')
    ax.text(0.05, 0.95, f"Epoch: {int(epoch)}", transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    ax.set_ylim(6, 8)


for idx in range(num_epochs, len(axes)):
    axes[idx].axis('off')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=5, frameon=True)
fig.tight_layout()
plt.savefig("streamfit_test_output/ra_vs_velocity_over_epochs.png", bbox_inches='tight')
plt.show()

In [ ]:
# plot dec vs velocity for every epoch in a single figure with subplots
if 'epoch_models' not in globals():
    raise RuntimeError("Run Cell 8 first to precompute epoch_models.")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()
for idx, epoch_data in enumerate(epoch_models):
    ax = axes[idx]

    dec_model = epoch_data['dec_model']
    v_model = epoch_data['v_model']
    dec_model_interp = epoch_data['dec_model_interp']
    v_model_interp = epoch_data['v_model_interp']
    valid = epoch_data['valid']
    epoch = epoch_data['epoch']

    ax.scatter(pc_coords[1], pc_coords[2], s=1, alpha=0.3, color='grey', label='Point cloud')
    ax.scatter(0, 7, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
    ax.errorbar(
        dec_data[valid], v_data[valid],
        xerr=dec_sigma[valid], yerr=v_sigma[valid],
        fmt='o-', label='Retained data', color='red'
    )
    ax.plot(dec_model, v_model, color='blue', linewidth=2, label='Model Streamline')
    ax.scatter(
        dec_model_interp[valid], v_model_interp[valid],
        s=25, label='Model at retained data arc lengths', color='blue', zorder=5
    )
    ax.scatter(
        dec_data[valid], v_data[valid],
        s=45, facecolor='none', edgecolor='cyan', linewidth=1.2,
        label='Retained data points', zorder=6
    )
    ax.set_xlabel('Dec Offset (arcsec)')
    ax.set_ylabel('Velocity (km/s)')
    ax.set_title('Dec vs Velocity')
    ax.text(0.05, 0.95, f"Epoch: {int(epoch)}", transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    ax.set_ylim(6, 8)
    ax.set_xlim(left=-11)

for idx in range(num_epochs, len(axes)):
    axes[idx].axis('off')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=5, frameon=True)
fig.tight_layout()
plt.savefig("streamfit_test_output/dec_vs_velocity_over_epochs.png", bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib.lines import Line2D
import numpy as np
# import wcs and coordinate utilities for KDE plotting
from astropy.wcs import WCS
from astropy.io import fits
from astropy import coordinates
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.convolution import Gaussian1DKernel, Gaussian2DKernel
from astropy.convolution import convolve
from velocity_tools import coordinate_offsets

# plot velocity vs radial distance for the final epoch
final_epoch_data = epoch_models[-1]

ra_model = final_epoch_data['ra_model']
dec_model = final_epoch_data['dec_model']
v_model = final_epoch_data['v_model']

radial_distance = jnp.sqrt(ra_model**2 + dec_model**2)

plt.figure(figsize=(8, 6))


# KDE of the velocity vs radial distance from the star
from scipy import stats
vlsr = 7.5

label_fs = 14
title_fs = 16
tick_fs = 12
legend_fs = 11

# --- KDE for velocity plot ---
## Choose limits for KDE
xmin, xmax = 0, 11    #arcsec
ymin, ymax = 6, 9      #km/s
# grid for kernel distribution
xx, yy = np.mgrid[xmin:xmax:100j, ymin:ymax:100j]
positions = np.vstack([xx.ravel(), yy.ravel()])

# Create a proper 2D header from celestial WCS
vel_hdu = fits.open('test_data/IRAS2A/D2CO_streamer_cluster_velocity.fits')[0]
vel_map = vel_hdu.data
vel_wcs = WCS(vel_hdu.header).celestial
vel_header_2d = vel_wcs.to_header()
vel_header_2d['NAXIS'] = 2
vel_header_2d['NAXIS1'] = vel_map.shape[1]
vel_header_2d['NAXIS2'] = vel_map.shape[0]
iras2a_c = SkyCoord("3h28m55.569s", "+31d14m37.025s", frame='fk5')

results = coordinate_offsets.generate_offsets(
    vel_header_2d,
    iras2a_c.ra,
    iras2a_c.dec,
    pa_angle=0*u.deg,
    inclination=0*u.deg
)
rproj_data = results.r.to(u.arcsec)
vlos_data = vel_map * u.km/u.s

good = np.isfinite(rproj_data * vlos_data)
values = np.vstack([rproj_data[good].value, vlos_data[good].value])

# KDE calculation
kernel = stats.gaussian_kde(values)
zz = np.reshape(kernel(positions).T, xx.shape)
zz /= zz.max()
kde_levels = np.append(np.exp(-0.5 * np.arange(1.0, 2.1, 0.5)**2)[::-1], [1.0])

# plot
fig, ax = plt.subplots(figsize=(6.5*1.3, 4*1.3))

ax.contourf(xx, yy, zz, levels=kde_levels, cmap='Greys', vmin=0, vmax=1.2)

# central star marker
plt.scatter(0, 7.5, marker='*', s=100, color='yellow', edgecolor='black', zorder=10, label='Central Source')

# plot the data points with error bars
data_handle = plt.errorbar(jnp.sqrt(ra_data**2 + dec_data**2), v_data,
                           xerr=jnp.sqrt(ra_sigma**2 + dec_sigma**2),
                           yerr=v_sigma,
                           fmt='o',
                           color='red',
                           label='Extracted 1D Streamline')

# plot the model
model_handle, = plt.plot(radial_distance, v_model, color='blue', label='Model Streamline', zorder=5)
# plot the model points at the retained data arc lengths
plt.scatter(jnp.sqrt(final_epoch_data['ra_model_interp'][final_epoch_data['valid']]**2 + final_epoch_data['dec_model_interp'][final_epoch_data['valid']]**2),
            final_epoch_data['v_model_interp'][final_epoch_data['valid']],
            s=25, color='blue', label='Model at retained data arc lengths', zorder=6)
# # custom darker legend marker for point cloud
# pc_legend = Line2D([0], [0], marker='.', color='w', label='Point cloud',
#                    markerfacecolor='grey', markeredgecolor='grey',
#                    markersize=1, alpha=1)

# horizontal line for systemic velocity
plt.axhline(7.5, color='black', linestyle='--', label='Systemic Velocity', zorder=3)
# epoch title
plt.title(f'Epoch: 300')

plt.xlabel('Projected Distance from Source (arcsec)')
plt.ylabel('Velocity (km/s)')
plt.xlim(-1, 11)
plt.ylim(7, 7.8)

# custom legend
plt.legend(handles=[data_handle, model_handle])

plt.savefig("streamfit_test_output/velocity_vs_radial_distance_final_epoch.png", bbox_inches='tight')

plt.show()